In [1]:
import os
import sys
sys.path.append(os.path.abspath('/home/pamikem/git_repos/performances_prediction_of_asset_allocations'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import MinMaxScaler

from predict_perf_allocation.soft_dtw import SoftDTW, SquaredEuclidean

In [12]:
X_train = pd.read_csv('../data/raw/X_train.csv',index_col='ROW_ID')
RET_COLS = [f'RET_{i}' for i in range(20, 0, -1)]       # t-20 ... t-1
VOL_COLS = [f'SIGNED_VOLUME_{i}' for i in range(20, 0, -1)]
X_LABELS = [f't-{i}' for i in range(20, 0, -1)]

In [13]:
# Drop rows with NA values in RET_COLS and VOL_COLS
rows_not_na = X_train.notna().all(axis=1)
X_train = X_train[rows_not_na].copy()

# X_train[RET_COLS] = np.sign(X_train[RET_COLS])


X_train.reset_index(drop=True, inplace=True)

scaler = MinMaxScaler()
# scaler.fit(X_train[RET_COLS].values.reshape(-1, 1))
# X_train[RET_COLS] = scaler.transform(X_train[RET_COLS].values.reshape(-1, 1)).reshape(-1, len(RET_COLS))

scaler.fit(X_train[VOL_COLS].values.reshape(-1, 1))
X_train[VOL_COLS] = scaler.transform(X_train[VOL_COLS].values.reshape(-1, 1)).reshape(-1, len(VOL_COLS))

### Computation between two time-series

In [ ]:
X1 = X_train.loc[1:3, RET_COLS].values#.reshape(1, -1)
X2 = X_train.loc[2:4, RET_COLS].values#.reshape(1, -1)

In [29]:
D = SquaredEuclidean(X1, X2)
sdtw = SoftDTW(D, gamma=1.0)
# gradient w.r.t. D, shape = [m, n], which is also the expected alignment matrix
value = sdtw.compute()

In [30]:
# gradient w.r.t. D, shape = [m, n], which is also the expected alignment matrix
E = sdtw.grad()
# gradient w.r.t. X, shape = [m, d]
G = D.jacobian_product(E)

### Pairwise computation

In [14]:
X1 = X_train.loc[1:4, RET_COLS].to_numpy()[:,:, np.newaxis]
X2 = X_train.loc[1:4, VOL_COLS].to_numpy()[:,:, np.newaxis]
X = np.concatenate((X1, X2), axis=2)
X.shape

(4, 20, 2)

In [17]:
gamma = 0.1
res = SoftDTW.pairwise(X, gamma=gamma, lambda_=1.0)
kernel_res = np.exp(-res / gamma)

In [18]:
kernel_res

array([[1.00172548, 0.97090766, 0.98346829, 0.98453229],
       [0.97090766, 1.00172634, 0.99312832, 0.99585893],
       [0.98346829, 0.99312832, 1.00172613, 0.99668076],
       [0.98453229, 0.99585893, 0.99668076, 1.00172645]])